# Verisift — Stage 4B: Text Preprocessing & Split Design
## Preparing Resume–Job Description Pairs for Machine Learning & Deep Learning

**Project Stage:** Stage 4B (Text Preprocessing & Validation Split Design)  
**Deliverable Notebook:** `notebooks/02_text_preprocessing.ipynb`  
**Upstream Exploration:** `notebooks/01_dataset_exploration.ipynb`  
**Dataset Files:** `train.csv` (Training Corpus), `test.csv` (Holdout Evaluation Corpus)  

---

### Table of Contents
1. [Project Context & Preprocessing Objectives](#1.-Project-Context-&-Preprocessing-Objectives)
2. [Environment Setup & Data Loading](#2.-Environment-Setup-&-Data-Loading)
3. [Target Label Validation & Explicit Mapping](#3.-Target-Label-Validation-&-Explicit-Mapping)
4. [Conservative Text Normalization Architecture](#4.-Conservative-Text-Normalization-Architecture)
5. [Before-and-After Comparison & Length Impact](#5.-Before-and-After-Comparison-&-Length-Impact)
6. [Duplicate Rows & Conflicting-Label Analysis](#6.-Duplicate-Rows-&-Conflicting-Label-Analysis)
7. [Grouped Split Design (Preventing Validation Leakage)](#7.-Grouped-Split-Design-(Preventing-Validation-Leakage))
8. [Reusable Preprocessing Module & Technical API](#8.-Reusable-Preprocessing-Module-&-Technical-API)
9. [Preprocessing Metadata & Artifacts Persistence](#9.-Preprocessing-Metadata-&-Artifacts-Persistence)
10. [Summary, Viva Defense & Next Steps](#10.-Summary,-Viva-Defense-&-Next-Steps)

---


## 1. Project Context & Preprocessing Objectives

### The Verisift Candidate-Job Matching Task
In Stages 1 through 3 of **Verisift**, we established the platform infrastructure:
- **Stage 1:** Recruiter authentication and JWT security.
- **Stage 2:** Job description posting, structured skill tagging, and management.
- **Stage 3:** Candidate pipeline, document upload, and background text extraction from PDF and DOCX files.

In **Stage 4**, our objective is to automate the matching decision: given a candidate's extracted resume text ($R$) and a job description ($J$), determine their degree of fit into one of three empirical categories:
- **`No Fit` (Class 0):** The candidate lacks the fundamental qualifications, technical skills, or domain experience specified in the job posting.
- **`Potential Fit` (Class 1):** The candidate possesses adjacent, transferable skills or meets a subset of core requirements; warrants manual recruiter review.
- **`Good Fit` (Class 2):** Strong semantic alignment across technical stack, responsibilities, and seniority.

### Why Careful Text Preprocessing is Critical
Raw text originating from PDF extraction (Stage 3) and web job postings contains several noise sources identified during Stage 4A:
1. **Extraction Artifacts:** Corrupt font encodings (`\ufffd`), inconsistent bullet markers (`•`, `▪`, `\u25cf`), and smart quotes (`“`, `”`, `‘`, `’`).
2. **Structural Inconsistencies:** Arbitrary line breaks, Windows CRLF line endings (`\r\n`), and runs of multiple blank lines.
3. **High-Signal Technical Tokens:** Modern tech resumes rely on punctuation-sensitive tokens:
   - Languages: `C++`, `C#`, `PL/SQL`, `R`
   - Frameworks & Runtimes: `.NET`, `Node.js`, `Vue.js`, `React.js`
   - Protocols & Systems: `TCP/IP`, `CI/CD`, `SQL/NoSQL`, `REST/GraphQL`
   - Numeric Qualifications: `5+ years`, `Python 3.12`, `AWS S3`

### Preprocessing Philosophy: Conservative & Transformer-Compatible
Aggressive traditional NLP techniques—such as lowercasing everything, removing stopwords, and applying Porter stemming—are **harmful** for modern Deep Learning and Transformer models:
- **Stopwords carry critical context:** Removing words like *"not"*, *"without"*, or *"with"* flips sentence polarity (e.g., *"not proficient in Kubernetes"* becomes *"proficient Kubernetes"*).
- **Stemming destroys technical jargon:** Truncating words causes fatal collisions (e.g., *"React"* $\rightarrow$ *"react"*, *"Spring"* $\rightarrow$ *"spring"*).
- **Case carries acronym signal:** Differentiating `AWS` (Amazon Web Services) from `aws` or `Go` (programming language) from `go` (verb).

Therefore, our preprocessing pipeline implements **conservative, structural normalization** that is 100% compatible with both baseline Bag-of-Words/TF-IDF models and modern Sentence Transformers / Deep Learning encoders.

> **Evaluation Rule:** `test.csv` is strictly reserved for holdout evaluation. No learned vocabulary, vectorizer, or label encoders will be fit on the test set.


## 2. Environment Setup & Data Loading

We import necessary Python libraries, set random seeds, and dynamically resolve file paths. The loading mechanism operates seamlessly whether the notebook is executed from the repository root or from within the `notebooks/` directory.

We load `train.csv` for processing and inspect `test.csv` strictly in read-only mode for structural verification.


In [1]:
import os
import sys
import json
import re
import unicodedata
from pathlib import Path

# Core data manipulation
import pandas as pd
import numpy as np

# Cross-validation
from sklearn.model_selection import GroupKFold

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display & reproducibility
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Dynamic path resolution: works both from root and from notebooks/ directory
cwd = Path.cwd()
if (cwd / 'train.csv').exists():
    project_root = cwd
elif (cwd.parent / 'train.csv').exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not locate train.csv and test.csv in current directory or parent directory.")

train_path = (project_root / 'train.csv').resolve()
test_path = (project_root / 'test.csv').resolve()

print(f"Project root resolved to: {project_root}")
print(f"Train path: {train_path} (Exists: {train_path.exists()})")
print(f"Test path:  {test_path} (Exists: {test_path.exists()})")

# Load training dataset
df_train = pd.read_csv(train_path)
print(f"\nLoaded df_train: {df_train.shape[0]:,} rows, {df_train.shape[1]} columns")

# Read-only verification of test dataset (no modification, no transformation)
df_test = pd.read_csv(test_path)
print(f"Loaded df_test (read-only verification): {df_test.shape[0]:,} rows, {df_test.shape[1]} columns")

# Technical Assertions for Schema Integrity
REQUIRED_COLUMNS = ['resume_text', 'job_description_text', 'label']
assert list(df_train.columns) == REQUIRED_COLUMNS, f"df_train columns mismatch: {df_train.columns.tolist()}"
assert list(df_test.columns) == REQUIRED_COLUMNS, f"df_test columns mismatch: {df_test.columns.tolist()}"
assert len(df_train) == 6241, f"Expected 6,241 train rows, found {len(df_train)}"
assert len(df_test) == 1759, f"Expected 1,759 test rows, found {len(df_test)}"
assert df_train['resume_text'].str.strip().ne('').all(), "Found empty resume_text in train"
assert df_train['job_description_text'].str.strip().ne('').all(), "Found empty job_description_text in train"

print("All structural schema assertions PASSED successfully.")


Project root resolved to: C:\Users\Shlok Chorghe\Desktop\DL_CP
Train path: C:\Users\Shlok Chorghe\Desktop\DL_CP\train.csv (Exists: True)
Test path:  C:\Users\Shlok Chorghe\Desktop\DL_CP\test.csv (Exists: True)



Loaded df_train: 6,241 rows, 3 columns
Loaded df_test (read-only verification): 1,759 rows, 3 columns
All structural schema assertions PASSED successfully.


## 3. Target Label Validation & Explicit Mapping

### The Importance of Explicit Ordinal Ordering
The target column `label` contains three string values: `'No Fit'`, `'Potential Fit'`, and `'Good Fit'`.

Default alphabetic label encoding (`LabelEncoder` in scikit-learn) assigns integers based on ASCII alphabetical sorting:
- `'Good Fit'` $\rightarrow 0$
- `'No Fit'` $\rightarrow 1$
- `'Potential Fit'` $\rightarrow 2$

This alphabetical assignment is completely arbitrary and dangerous:
1. It is non-monotonic: `Good Fit` (0) and `Potential Fit` (2) are split by `No Fit` (1).
2. For regression, ordinal classification, ranking, or thresholding, the distance $|y_i - y_j|$ would be semantically meaningless.
3. Confusion matrices and class reports would be counter-intuitive.

### Our Explicit, Reversible Mapping
We define a monotonically increasing scale of candidate suitability:
$$\text{No Fit (0)} \quad < \quad \text{Potential Fit (1)} \quad < \quad \text{Good Fit (2)}$$

We implement bidirectional dictionaries:
- `LABEL2ID`: Maps string label to integer ID.
- `ID2LABEL`: Maps integer ID back to human-readable string label.


In [2]:
# 1. Inspect ground truth label counts
raw_counts = df_train['label'].value_counts()
print("=== RAW TRAINING LABELS ===")
print(raw_counts)

# 2. Define explicit, monotonic bidirectional mapping
LABEL2ID = {
    'No Fit': 0,
    'Potential Fit': 1,
    'Good Fit': 2
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# 3. Create target integer column
df_train['label_id'] = df_train['label'].map(LABEL2ID)

# 4. Rigorous assertions for label integrity
assert set(df_train['label'].unique()) == set(LABEL2ID.keys()), "Unexpected labels found in train set!"
assert df_train['label_id'].isna().sum() == 0, "Null values introduced during label mapping!"
assert df_train['label_id'].map(ID2LABEL).equals(df_train['label']), "Label mapping is not strictly reversible!"
assert set(df_test['label'].unique()) == set(LABEL2ID.keys()), "Test set contains unseen labels!"

# 5. Display verification summary
label_mapping_summary = pd.DataFrame({
    'String Label': list(LABEL2ID.keys()),
    'Assigned Integer ID': list(LABEL2ID.values()),
    'Suitability Level': ['Unsuitable / Rejection', 'Partial / Review Warranted', 'Strong Match / Interview'],
    'Train Frequency': [raw_counts[l] for l in LABEL2ID.keys()],
    'Train Percentage': [(raw_counts[l] / len(df_train) * 100).round(2) for l in LABEL2ID.keys()]
})

print("\n=== EXPLICIT LABEL MAPPING SPECIFICATION ===")
display(label_mapping_summary)


=== RAW TRAINING LABELS ===
label
No Fit           3143
Potential Fit    1556
Good Fit         1542
Name: count, dtype: int64

=== EXPLICIT LABEL MAPPING SPECIFICATION ===


,String Label,Assigned Integer ID,Suitability Level,Train Frequency,Train Percentage
0,No Fit,0,Unsuitable / Rejection,3143,50.36
1,Potential Fit,1,Partial / Review Warranted,1556,24.93
2,Good Fit,2,Strong Match / Interview,1542,24.71


## 4. Conservative Text Normalization Architecture

### Design of the Normalization Pipeline
To maximize downstream model compatibility (for both Bag-of-Words / TF-IDF and Transformer embeddings), our `clean_text` function executes seven modular stages:

1. **Unicode Compatibility Normalization (NFKC):**
   Decomposes and standardizes Unicode characters (e.g. ligature formatting, wide characters) while preserving vital diacritics and accented characters (e.g., *résumé*, *naïve*).
2. **Replacement Character Handling (`\ufffd`):**
   The character `\ufffd` appears in documents where PDF font extraction failed. We replace `\ufffd` with a whitespace delimiter rather than allowing it to corrupt tokenizers.
3. **Typographic Symbol Normalization:**
   - Curly quotes (`“`, `”`, `‘`, `’`) are converted to standard ASCII quotes (`"`, `'`).
   - Dashes (em-dash `—`, en-dash `–`, figure dash `‒`) are normalized to standard hyphens (`-`).
   - Bullet symbols (`•`, `▪`, `\u25cf`, etc.) are converted to standard bullet markdown `- ` so hierarchical skill and responsibility lists are cleanly delineated.
4. **Preservation of Technical Punctuation:**
   We strictly protect programming language names, frameworks, and versions. We do **not** apply regex replacements that strip punctuation indiscriminately (e.g., `re.sub(r'[^a-zA-Z]', ' ', text)` is strictly prohibited as it destroys `C++`, `C#`, `.NET`, `Node.js`).
5. **Line Ending Standardization:**
   Converts Windows CRLF (`\r\n`) and old Mac CR (`\r`) to standard Unix LF (`\n`).
6. **Control Character Stripping:**
   Removes non-printable ASCII control bytes (`[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]`) while retaining standard formatting newlines (`\n`) and tabs (`\t`).
7. **Whitespace & Paragraph Consolidation:**
   Collapses multiple inline tabs and spaces to a single space per line. Collapses runs of three or more newlines to a maximum of two (`\n\n`), preserving clean paragraph boundaries without unbounded blank space.


In [3]:
def clean_text(text: str) -> str:
    """
    Applies conservative, semantic-preserving text normalization.
    
    Safe for both traditional TF-IDF feature extractors and modern
    Sentence Transformers / Deep Learning language models.
    """
    if not isinstance(text, str) or not text:
        return ""
    
    # 1. Unicode normalization (NFKC)
    text = unicodedata.normalize('NFKC', text)
    
    # 2. Replace Unicode replacement character (extraction artifact) with space
    text = text.replace('\ufffd', ' ')
    
    # 3. Standardize typographic quotes and apostrophes
    text = text.replace('\u201c', '"').replace('\u201d', '"').replace('\u2018', "'").replace('\u2019', "'")
    
    # 4. Standardize dashes to standard hyphen
    text = text.replace('\u2014', '-').replace('\u2013', '-').replace('\u2012', '-')
    
    # 5. Standardize bullet characters to uniform hyphen marker
    bullet_regex = r'[\u2022\u2023\u25E6\u2043\u2219\u25CB\u25CF\u25AA\u25AB\uF0B7]'
    text = re.sub(bullet_regex, ' - ', text)
    
    # 6. Standardize line endings to standard Unix LF (\n)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    
    # 7. Strip non-printable ASCII control characters (preserving \n and \t)
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
    
    # 8. Normalize inline horizontal whitespace per line
    lines = []
    for line in text.split('\n'):
        cleaned_line = re.sub(r'[ \t]+', ' ', line).strip()
        lines.append(cleaned_line)
    
    # 9. Collapse excessive newlines (max 2 consecutive newlines to preserve paragraphs)
    normalized = '\n'.join(lines)
    normalized = re.sub(r'\n{3,}', '\n\n', normalized).strip()
    
    return normalized

# Comprehensive Unit Test Suite
print("Running unit assertions on clean_text() edge cases...")

# Test 1: Technical programming languages & frameworks
sample_tech = "Proficient in C++, C#, .NET 6.0, Node.js, and CI/CD pipelines."
assert clean_text(sample_tech) == sample_tech, "Failed to preserve technical tokens!"

# Test 2: Unicode replacement character cleanup
sample_replacement = "QUALIFICATIONS\ufffd Over two years' experience"
assert clean_text(sample_replacement) == "QUALIFICATIONS Over two years' experience", "Failed replacement char handling!"

# Test 3: Typographic quotes and dashes
sample_quotes = "\u201cSenior Software Engineer\u201d \u2013 Python \u2018developer\u2019"
expected_quotes = '"Senior Software Engineer" - Python ' + chr(39) + 'developer' + chr(39)
assert clean_text(sample_quotes) == expected_quotes, "Failed quote/dash normalization!"

# Test 4: Bullets and excessive whitespace
sample_bullets = "Skills:\n\t\u2022 Docker\n\n\n\t\u2022 Kubernetes\n\n\n\n\u2022 Terraform"
expected_bullets = "Skills:\n- Docker\n\n- Kubernetes\n\n- Terraform"
assert clean_text(sample_bullets) == expected_bullets, f"Failed bullet normalization: got {repr(clean_text(sample_bullets))}"

# Test 5: Accented characters
sample_accents = "R\u00e9sum\u00e9 of candidate with na\u00efve Bayes experience."
assert clean_text(sample_accents) == sample_accents, "Failed to preserve accented characters!"

# Test 6: Empty and non-string inputs
assert clean_text("") == "", "Empty string should return empty string"
assert clean_text(None) == "", "None input should return empty string"

print("All 6 unit test assertions PASSED successfully!")


Running unit assertions on clean_text() edge cases...
All 6 unit test assertions PASSED successfully!


## 5. Before-and-After Comparison & Length Impact

We apply `clean_text` to both `resume_text` and `job_description_text` across the training corpus. We create:
- `resume_text_clean`
- `job_description_text_clean`

We inspect the quantitative impact on character count and word count, and present sanitized before-and-after excerpts. To respect privacy, any potential personal contact details (such as emails or telephone numbers) are redacted using regex filters before display.


In [4]:
# Apply normalization to train dataset
df_train['resume_text_clean'] = df_train['resume_text'].apply(clean_text)
df_train['job_description_text_clean'] = df_train['job_description_text'].apply(clean_text)

# Technical assertion: ensure cleaned texts are non-empty
assert df_train['resume_text_clean'].str.strip().ne('').all(), "Cleaned resume text contains empty records!"
assert df_train['job_description_text_clean'].str.strip().ne('').all(), "Cleaned job text contains empty records!"

# Calculate character and word count deltas
df_train['res_char_raw'] = df_train['resume_text'].str.len()
df_train['res_char_clean'] = df_train['resume_text_clean'].str.len()
df_train['res_char_diff'] = df_train['res_char_raw'] - df_train['res_char_clean']

df_train['res_word_raw'] = df_train['resume_text'].apply(lambda x: len(x.split()))
df_train['res_word_clean'] = df_train['resume_text_clean'].apply(lambda x: len(x.split()))
df_train['res_word_diff'] = df_train['res_word_raw'] - df_train['res_word_clean']

df_train['job_char_raw'] = df_train['job_description_text'].str.len()
df_train['job_char_clean'] = df_train['job_description_text_clean'].str.len()
df_train['job_char_diff'] = df_train['job_char_raw'] - df_train['job_char_clean']

df_train['job_word_raw'] = df_train['job_description_text'].apply(lambda x: len(x.split()))
df_train['job_word_clean'] = df_train['job_description_text_clean'].apply(lambda x: len(x.split()))
df_train['job_word_diff'] = df_train['job_word_raw'] - df_train['job_word_clean']

# Summary table of length changes
diff_summary = pd.DataFrame([
    {
        'Text Field': 'Resume Characters',
        'Raw Mean': df_train['res_char_raw'].mean().round(1),
        'Clean Mean': df_train['res_char_clean'].mean().round(1),
        'Avg Reduction': df_train['res_char_diff'].mean().round(1),
        '% Reduction': ((df_train['res_char_diff'].mean() / df_train['res_char_raw'].mean()) * 100).round(2),
        'Max Reduction': df_train['res_char_diff'].max()
    },
    {
        'Text Field': 'Resume Words',
        'Raw Mean': df_train['res_word_raw'].mean().round(1),
        'Clean Mean': df_train['res_word_clean'].mean().round(1),
        'Avg Reduction': df_train['res_word_diff'].mean().round(1),
        '% Reduction': ((df_train['res_word_diff'].mean() / df_train['res_word_raw'].mean()) * 100).round(2),
        'Max Reduction': df_train['res_word_diff'].max()
    },
    {
        'Text Field': 'Job Characters',
        'Raw Mean': df_train['job_char_raw'].mean().round(1),
        'Clean Mean': df_train['job_char_clean'].mean().round(1),
        'Avg Reduction': df_train['job_char_diff'].mean().round(1),
        '% Reduction': ((df_train['job_char_diff'].mean() / df_train['job_char_raw'].mean()) * 100).round(2),
        'Max Reduction': df_train['job_char_diff'].max()
    },
    {
        'Text Field': 'Job Words',
        'Raw Mean': df_train['job_word_raw'].mean().round(1),
        'Clean Mean': df_train['job_word_clean'].mean().round(1),
        'Avg Reduction': df_train['job_word_diff'].mean().round(1),
        '% Reduction': ((df_train['job_word_diff'].mean() / df_train['job_word_raw'].mean()) * 100).round(2),
        'Max Reduction': df_train['job_word_diff'].max()
    }
])

print("=== QUANTITATIVE IMPACT OF NORMALIZATION ===")
display(diff_summary.set_index('Text Field'))

# Visual Before-and-After Comparison on Sanitized Samples
def sanitize_display(text: str) -> str:
    """Redacts potential PII patterns for safe visual inspection."""
    t = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '[REDACTED_EMAIL]', text)
    t = re.sub(r'(\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', '[REDACTED_PHONE]', t)
    return t

print("\n=== SANITIZED QUALITATIVE BEFORE-AND-AFTER INSPECTION ===")
sample_idx = 0
sample_raw_res = sanitize_display(df_train.iloc[sample_idx]['resume_text'])
sample_clean_res = sanitize_display(df_train.iloc[sample_idx]['resume_text_clean'])

print("--- RAW RESUME SAMPLE (FIRST 250 CHARS) ---")
print(repr(sample_raw_res[:250]))

print("\n--- CLEANED RESUME SAMPLE (FIRST 250 CHARS) ---")
print(repr(sample_clean_res[:250]))


=== QUANTITATIVE IMPACT OF NORMALIZATION ===


,Raw Mean,Clean Mean,Avg Reduction,% Reduction,Max Reduction
Text Field,,,,,
Resume Characters,5804.4,5792.0,12.4,0.21,440
Resume Words,708.1,708.5,-0.4,-0.06,0
Job Characters,2731.3,2728.4,2.9,0.11,46
Job Words,371.5,371.5,0.0,0.00,0



=== SANITIZED QUALITATIVE BEFORE-AND-AFTER INSPECTION ===
--- RAW RESUME SAMPLE (FIRST 250 CHARS) ---
'SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.\nHigh'

--- CLEANED RESUME SAMPLE (FIRST 250 CHARS) ---
'SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.\nHigh'


## 6. Duplicate Rows & Conflicting-Label Analysis

### Investigation of Repeated Pairs in the Training Corpus
During Stage 4A, we uncovered 7 pairs of identical `(resume_text, job_description_text)` in `train.csv` (spanning 14 rows):
- **1 Pair (2 rows):** Identical labels (`['No Fit', 'No Fit']`). This is an exact full-row duplicate.
- **6 Pairs (12 rows):** **Contradictory labels** (`No Fit` paired with `Good Fit` in 4 cases; `No Fit` paired with `Potential Fit` in 2 cases).

### Why Contradictory Labels Harm Supervised Learning
When the exact same input features $(R, J)$ are mapped to contradictory labels ($y=0$ and $y=2$):
1. **Conflicting Gradient Signals:** Cross-entropy loss heavily penalizes the model regardless of what it predicts on that pair.
2. **Artificial Loss Floor:** The model can never achieve lower training loss because the empirical Bayes error on those pairs is high.
3. **Subjective Annotation Noise:** These pairs represent borderline candidates where human recruiters or crowd-annotators disagreed.

### Transparent Treatment Strategy
We do **not** silently drop records. We explicitly document two views:
1. **`df_train` (Full Corpus):** The original 6,241 rows preserved in memory with boolean flags:
   - `is_exact_duplicate`: Identifies the 1 pure duplicate row.
   - `is_conflicting_pair`: Identifies the 12 rows with conflicting annotations.
2. **`df_train_dedup` (Cleaned Training View):**
   - The 1 exact duplicate row is dropped (1 row).
   - The 6 ambiguous conflicting pairs are removed (12 rows).
   - Clean training size: $6,241 - 13 = 6,228$ rows.
   - This provides clean, unambiguous supervisory signal for subsequent model training.

> **Crucial Rule:** `test.csv` has **0 duplicate pairs** and is **never modified**.


In [5]:
# Flag duplicate pairs in training set
pair_cols = ['resume_text_clean', 'job_description_text_clean']

# Find all occurrences of repeated pairs
dup_pair_mask = df_train.duplicated(subset=pair_cols, keep=False)
exact_dup_mask = df_train.duplicated(subset=['resume_text_clean', 'job_description_text_clean', 'label'], keep='first')

df_train['is_duplicate_pair'] = dup_pair_mask
df_train['is_exact_duplicate'] = exact_dup_mask

# Identify conflicting pairs
conflicting_indices = []
conflict_rows_info = []

for (r, j), group in df_train.groupby(pair_cols):
    if len(group) > 1:
        labels = group['label'].tolist()
        indices = group.index.tolist()
        is_conflict = len(set(labels)) > 1
        
        conflict_rows_info.append({
            'Pair Occurrences': len(group),
            'Assigned Labels': labels,
            'Row Indices': indices,
            'Has Conflicting Labels': is_conflict,
            'Resume Snippet': r[:50].replace('\n', ' ') + '...',
            'Job Snippet': j[:50].replace('\n', ' ') + '...'
        })
        if is_conflict:
            conflicting_indices.extend(indices)

conflict_df = pd.DataFrame(conflict_rows_info)
print("=== DUPLICATE & CONFLICTING PAIRS IN TRAINING SET ===")
display(conflict_df)

# Flag conflicting pairs
df_train['is_conflicting_pair'] = df_train.index.isin(conflicting_indices)

# Construct cleaned training view for model experiments
exact_dup_indices = df_train[df_train['is_exact_duplicate']].index.tolist()
drop_indices = sorted(list(set(exact_dup_indices + conflicting_indices)))

df_train_dedup = df_train.drop(index=drop_indices).copy().reset_index(drop=True)

print(f"\n--- DEDUPLICATION & CONFLICT RESOLUTION SUMMARY ---")
print(f"Original df_train records:           {len(df_train):,}")
print(f"Exact duplicate rows dropped:        {len(exact_dup_indices)} (index: {exact_dup_indices})")
print(f"Conflicting label pairs dropped:     {len(conflicting_indices)} rows across 6 ambiguous pairs")
print(f"Total rows removed for clean view:   {len(drop_indices)}")
print(f"Cleaned training view (df_train_dedup): {len(df_train_dedup):,} records")

# Assertions
assert len(df_train_dedup) == 6241 - 13, f"Expected 6,228 clean rows, got {len(df_train_dedup)}"
assert df_train_dedup.duplicated(subset=pair_cols).sum() == 0, "Cleaned training view still contains duplicate pairs!"


=== DUPLICATE & CONFLICTING PAIRS IN TRAINING SET ===


,Pair Occurrences,Assigned Labels,Row Indices,Has Conflicting Labels,Resume Snippet,Job Snippet
0,2,"[No Fit, Potential Fit]","[520, 4493]",True,Professional SummaryBusiness Intelligence Cons...,"ROR Partners is a rapidly growing, client-driv..."
1,2,"[No Fit, Good Fit]","[377, 5904]",True,ProfileOver 4 Years of Business Intelligence /...,Role - Business Analyst - Mobile Location - Lo...
2,2,"[No Fit, Good Fit]","[381, 5153]",True,QUALIFICATIONS - Over two years' experience of...,Calling all data nerds who love finance! If yo...
3,2,"[No Fit, Good Fit]","[46, 4920]",True,SummaryDynamic and motivated IT professional w...,Business Analyst Location: Remote.Duration: 12...
4,2,"[No Fit, Potential Fit]","[312, 3707]",True,SummaryHighly motivated Sales Associate with e...,Alexander Technology Group is currently exclus...
5,2,"[No Fit, No Fit]","[848, 2326]",False,SummaryHighly proficient Business Analyst with...,"The Senior Manager, Data Architecture & Data E..."
6,2,"[No Fit, Good Fit]","[585, 4889]",True,SummaryPursue a challenging and rewarding posi...,Cloud and Things is a leading provider of IT s...



--- DEDUPLICATION & CONFLICT RESOLUTION SUMMARY ---
Original df_train records:           6,241
Exact duplicate rows dropped:        1 (index: [2326])
Conflicting label pairs dropped:     12 rows across 6 ambiguous pairs
Total rows removed for clean view:   13
Cleaned training view (df_train_dedup): 6,228 records


## 7. Grouped Split Design (Preventing Validation Leakage)

### The Structural Reality of the Test Set
In Stage 4A, cross-split analysis revealed that:
1. **0 of the 71 test job descriptions** appear in `train.csv` (100% disjoint).
2. **476 of the 477 test resumes** exist in `train.csv` (candidate pool is evaluated against new jobs).

This structure directly reflects the production reality of the **Verisift** web platform:
- The recruiter platform maintains a persistent talent pool of stored candidate resumes.
- When a recruiter creates a **new job posting**, the system must score and rank the candidate database against that **previously unseen job**.

### Why Random K-Fold Splitting Causes Validation Leakage
If we apply standard random cross-validation (`KFold` or `train_test_split`):
- Pairs evaluated against the **same job description** will be split across both the training fold and the validation fold.
- The model can memorize job-specific terms, company names, or exact phrasing present in the training fold.
- This creates **validation leakage**: local cross-validation yields falsely inflated F1 scores, but when the model is tested on `test.csv` (which contains 100% unseen jobs), performance dramatically drops.

### The Solution: 5-Fold `GroupKFold` on Job Descriptions
We partition local cross-validation using **`GroupKFold`**, where groups are defined by the unique `job_description_text_clean`:
- Every job description (and all candidate pairs evaluated against it) is allocated exclusively to either the training folds or the validation fold.
- The intersection of jobs between training and validation is **strictly empty ($\emptyset$)**.

### Trade-offs & Evaluation Limitations
- **Advantage:** Guarantees zero job-leakage; faithfully simulates holdout test performance on unseen roles.
- **Trade-off:** Candidate resumes appear across different folds (evaluated against different jobs).
- **Fold Size Variation:** Because each job posting attracts a slightly different number of candidates, fold sample sizes and class distributions may vary marginally across folds.


In [6]:
# Define 5-Fold GroupKFold on cleaned training corpus
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

# Group strictly by clean job description
groups = df_train_dedup['job_description_text_clean']

# Assign fold column
df_train_dedup['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(df_train_dedup, df_train_dedup['label_id'], groups)):
    df_train_dedup.loc[val_idx, 'fold'] = fold

# Verification and disjoint assertion across all folds
fold_audit = []
for f in range(N_SPLITS):
    train_subset = df_train_dedup[df_train_dedup['fold'] != f]
    val_subset = df_train_dedup[df_train_dedup['fold'] == f]
    
    train_jobs = set(train_subset['job_description_text_clean'])
    val_jobs = set(val_subset['job_description_text_clean'])
    job_overlap = train_jobs.intersection(val_jobs)
    
    # Non-negotiable technical assertion: zero job overlap
    assert len(job_overlap) == 0, f"DATA LEAKAGE DETECTED in Fold {f}: {len(job_overlap)} shared jobs!"
    
    # Class distribution in validation fold
    val_dist = val_subset['label'].value_counts(normalize=True) * 100
    
    fold_audit.append({
        'Fold': f,
        'Train Pairs': len(train_subset),
        'Val Pairs': len(val_subset),
        'Train Unique Jobs': len(train_jobs),
        'Val Unique Jobs': len(val_jobs),
        'Job Overlap': len(job_overlap),
        'Val % No Fit': val_dist.get('No Fit', 0.0).round(1),
        'Val % Potential Fit': val_dist.get('Potential Fit', 0.0).round(1),
        'Val % Good Fit': val_dist.get('Good Fit', 0.0).round(1)
    })

fold_audit_df = pd.DataFrame(fold_audit)
print("=== 5-FOLD GROUPKFOLD DISJOINT VERIFICATION AUDIT ===")
display(fold_audit_df.set_index('Fold'))

# Global assertions
assert (df_train_dedup['fold'] >= 0).all(), "Unassigned folds detected!"
assert len(df_train_dedup['fold'].unique()) == N_SPLITS, f"Expected {N_SPLITS} folds!"
print("\nAll 5 validation folds are STRICTLY DISJOINT with ZERO job description leakage.")


=== 5-FOLD GROUPKFOLD DISJOINT VERIFICATION AUDIT ===


,Train Pairs,Val Pairs,Train Unique Jobs,Val Unique Jobs,Job Overlap,Val % No Fit,Val % Potential Fit,Val % Good Fit
Fold,,,,,,,,
0,4982,1246,224,56,0,52.3,28.1,19.6
1,4982,1246,224,56,0,49.5,24.7,25.8
2,4982,1246,224,56,0,52.4,22.6,25.0
3,4983,1245,224,56,0,47.2,23.2,29.6
4,4983,1245,224,56,0,50.3,26.2,23.5



All 5 validation folds are STRICTLY DISJOINT with ZERO job description leakage.


## 8. Reusable Preprocessing Module & Technical API

To ensure seamless integration between research notebooks (`02_text_preprocessing.ipynb`, `03_baseline_models.ipynb`, `04_deep_learning.ipynb`) and the live **Verisift backend application** (`backend/app/services`), we package the preprocessing logic into a standalone, object-oriented class: `TextPreprocessor`.

### API Capabilities of `TextPreprocessor`
- `preprocess_text(text: str) -> str`: Normalizes an individual string.
- `encode_labels(labels: Sequence[str]) -> np.ndarray`: Encodes string labels into monotonic integer IDs.
- `decode_labels(ids: Sequence[int]) -> list[str]`: Decodes integer IDs back to string labels.
- `preprocess_pair(resume: str, job: str) -> tuple[str, str]`: Normalizes a resume-job text pair for inference.
- `save_config(path: str)` / `load_config(path: str)`: Serializes configuration to ensure reproducible deployment.

We write the reusable module directly to `backend/app/services/text_preprocessor.py` so the web app can use the exact same preprocessing logic.


In [7]:
class TextPreprocessor:
    """
    Reusable, production-grade text preprocessor for the Verisift platform.
    Shares identical normalization and label mapping across training notebooks
    and production FastAPI inference endpoints.
    """
    LABEL2ID = {
        'No Fit': 0,
        'Potential Fit': 1,
        'Good Fit': 2
    }
    ID2LABEL = {0: 'No Fit', 1: 'Potential Fit', 2: 'Good Fit'}
    
    @classmethod
    def normalize(cls, text: str) -> str:
        """Normalizes text while preserving technical tokens and sentence structure."""
        if not isinstance(text, str) or not text:
            return ""
        
        # Unicode NFKC normalization
        text = unicodedata.normalize('NFKC', text)
        
        # Clean replacement character �
        text = text.replace('\ufffd', ' ')
        
        # Normalize typographic quotes and dashes
        text = text.replace('\u201c', '"').replace('\u201d', '"').replace('\u2018', "'").replace('\u2019', "'")
        text = text.replace('\u2014', '-').replace('\u2013', '-').replace('\u2012', '-')
        
        # Standardize bullets
        text = re.sub(r'[\u2022\u2023\u25E6\u2043\u2219\u25CB\u25CF\u25AA\u25AB\uF0B7]', ' - ', text)
        
        # Line endings
        text = text.replace('\r\n', '\n').replace('\r', '\n')
        
        # Strip non-printable control characters
        text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
        
        # Inline horizontal whitespace
        lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in text.split('\n')]
        
        # Collapse newlines
        return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()
    
    @classmethod
    def encode_label(cls, label: str) -> int:
        if label not in cls.LABEL2ID:
            raise ValueError(f"Unknown label '{label}'. Must be one of {list(cls.LABEL2ID.keys())}")
        return cls.LABEL2ID[label]
    
    @classmethod
    def decode_label(cls, label_id: int) -> str:
        if label_id not in cls.ID2LABEL:
            raise ValueError(f"Unknown label_id {label_id}. Must be one of {list(cls.ID2LABEL.keys())}")
        return cls.ID2LABEL[label_id]
    
    @classmethod
    def preprocess_pair(cls, resume_text: str, job_text: str) -> tuple[str, str]:
        return cls.normalize(resume_text), cls.normalize(job_text)

# Test the preprocessor class
test_res, test_job = TextPreprocessor.preprocess_pair(
    "  \u2022 Senior C++ & Python Engineer\n\n\n  ",
    "Requirements:\n\u2022 5+ years C++ experience."
)

assert test_res == "- Senior C++ & Python Engineer"
assert test_job == "Requirements:\n- 5+ years C++ experience."
assert TextPreprocessor.encode_label("Good Fit") == 2
assert TextPreprocessor.decode_label(2) == "Good Fit"

# Export reusable module to backend/app/services/text_preprocessor.py
backend_service_dir = project_root / 'backend' / 'app' / 'services'
backend_service_dir.mkdir(parents=True, exist_ok=True)
service_file = backend_service_dir / 'text_preprocessor.py'

service_code = '''"""
Text Preprocessor Service for Verisift.
Generated during Stage 4B: Text Preprocessing.
Ensures identical text normalization and label encoding across training and API endpoints.
"""
import re
import unicodedata
from typing import Tuple, Dict, List

class TextPreprocessor:
    LABEL2ID: Dict[str, int] = {
        'No Fit': 0,
        'Potential Fit': 1,
        'Good Fit': 2
    }
    ID2LABEL: Dict[int, str] = {0: 'No Fit', 1: 'Potential Fit', 2: 'Good Fit'}
    
    @classmethod
    def normalize(cls, text: str) -> str:
        if not isinstance(text, str) or not text:
            return ""
        text = unicodedata.normalize('NFKC', text)
        text = text.replace('\\ufffd', ' ')
        text = text.replace('\\u201c', '"').replace('\\u201d', '"').replace('\\u2018', "'").replace('\\u2019', "'")
        text = text.replace('\\u2014', '-').replace('\\u2013', '-').replace('\\u2012', '-')
        text = re.sub(r'[\\u2022\\u2023\\u25E6\\u2043\\u2219\\u25CB\\u25CF\\u25AA\\u25AB\\uF0B7]', ' - ', text)
        text = text.replace('\\r\\n', '\\n').replace('\\r', '\\n')
        text = re.sub(r'[\\x00-\\x08\\x0B\\x0C\\x0E-\\x1F\\x7F]', '', text)
        lines = [re.sub(r'[ \\t]+', ' ', line).strip() for line in text.split('\\n')]
        return re.sub(r'\\n{3,}', '\\n\\n', '\\n'.join(lines)).strip()

    @classmethod
    def encode_label(cls, label: str) -> int:
        return cls.LABEL2ID[label]

    @classmethod
    def decode_label(cls, label_id: int) -> str:
        return cls.ID2LABEL[label_id]

    @classmethod
    def preprocess_pair(cls, resume_text: str, job_text: str) -> Tuple[str, str]:
        return cls.normalize(resume_text), cls.normalize(job_text)
'''

with open(service_file, 'w', encoding='utf-8') as f:
    f.write(service_code)

print(f"Successfully exported reusable service to: {service_file.resolve()}")


Successfully exported reusable service to: C:\Users\Shlok Chorghe\Desktop\DL_CP\backend\app\services\text_preprocessor.py


## 9. Preprocessing Metadata & Artifacts Persistence

### Reproducibility Without Data Duplication
In accordance with project constraints:
1. We do **not** duplicate or store large 50 MB intermediate CSV files.
2. We do **not** touch or overwrite the original `train.csv` or `test.csv` files.
3. We save small, clearly defined, reproducible metadata configuration files in the `artifacts/` directory:
   - `artifacts/label_mapping.json`: Bidirectional label mappings and target class metadata.
   - `artifacts/preprocessing_config.json`: Documented normalization rules, sequence parameters, and GroupKFold configuration.


In [8]:
# Create artifacts directory
artifacts_dir = project_root / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

# 1. Save label mapping artifact
label_mapping_payload = {
    'label2id': LABEL2ID,
    'id2label': {str(k): v for k, v in ID2LABEL.items()},
    'classes': list(LABEL2ID.keys()),
    'num_classes': len(LABEL2ID)
}

label_mapping_file = artifacts_dir / 'label_mapping.json'
with open(label_mapping_file, 'w', encoding='utf-8') as f:
    json.dump(label_mapping_payload, f, indent=2)

print(f"Saved label mapping artifact to: {label_mapping_file.resolve()}")

# 2. Save preprocessing metadata configuration
preprocessing_config_payload = {
    'stage': '4B',
    'version': '1.0.0',
    'normalization_pipeline': {
        'unicode_normalization': 'NFKC',
        'replacement_character_handling': 'replace_with_space',
        'typographic_quotes': 'convert_to_standard_ascii',
        'dashes': 'convert_to_hyphen',
        'bullets': 'standardize_to_hyphen_marker',
        'line_endings': 'convert_to_unix_lf',
        'control_characters': 'strip_non_printable',
        'whitespace': 'collapse_horizontal_max_two_newlines',
        'lowercasing_applied': False,
        'stopwords_removed': False,
        'stemming_applied': False,
        'technical_tokens_preserved': ['C++', 'C#', '.NET', 'Node.js', 'CI/CD', 'SQL/NoSQL', 'versions']
    },
    'validation_split_design': {
        'method': 'GroupKFold',
        'n_splits': 5,
        'group_column': 'job_description_text_clean',
        'guaranteed_disjoint_jobs': True,
        'train_clean_samples': len(df_train_dedup),
        'dropped_exact_duplicates': len(exact_dup_indices),
        'dropped_conflicting_pairs': len(conflicting_indices)
    }
}

config_file = artifacts_dir / 'preprocessing_config.json'
with open(config_file, 'w', encoding='utf-8') as f:
    json.dump(preprocessing_config_payload, f, indent=2)

print(f"Saved preprocessing config to:    {config_file.resolve()}")

# 3. Verification of Original CSV Files Integrity
train_stat = os.stat(train_path)
test_stat = os.stat(test_path)

print(f"\n--- ORIGINAL CSV INTEGRITY AUDIT ---")
print(f"train.csv size on disk: {train_stat.st_size:,} bytes (Original: 53,437,783 bytes)")
print(f"test.csv  size on disk: {test_stat.st_size:,} bytes (Original: 15,178,234 bytes)")
assert train_stat.st_size == 53437783, "CRITICAL: train.csv size has changed!"
assert test_stat.st_size == 15178234, "CRITICAL: test.csv size has changed!"
print("Original CSV files remain 100% UNTOUCHED and PRESERVED.")


Saved label mapping artifact to: C:\Users\Shlok Chorghe\Desktop\DL_CP\artifacts\label_mapping.json
Saved preprocessing config to:    C:\Users\Shlok Chorghe\Desktop\DL_CP\artifacts\preprocessing_config.json

--- ORIGINAL CSV INTEGRITY AUDIT ---
train.csv size on disk: 53,437,783 bytes (Original: 53,437,783 bytes)
test.csv  size on disk: 15,178,234 bytes (Original: 15,178,234 bytes)
Original CSV files remain 100% UNTOUCHED and PRESERVED.


## 10. Summary, Viva Defense & Next Steps

### 10.1 Summary of Preprocessing Decisions
| Component | Decision Taken | Rationale & Architectural Benefit |
| :--- | :--- | :--- |
| **Label Mapping** | Monotonic: `No Fit (0) < Potential Fit (1) < Good Fit (2)` | Preserves natural candidate suitability scale; allows ordinal metrics and meaningful thresholding. |
| **Unicode & Artifacts** | NFKC normalization; replace `\ufffd` with space; normalize curly quotes & dashes | Cleans PDF extraction defects without deleting foreign candidate names or accents. |
| **Technical Tokens** | Explicitly preserved (`C++`, `C#`, `.NET`, `Node.js`, `AWS`, `Python 3.12`) | Prevents destruction of essential programming language names and version specifications. |
| **Stopwords & Case** | **Deliberately Preserved** (No lowercasing, No stopword removal, No stemming) | Critical for modern Sentence Transformers / Deep Learning models that depend on grammatical context and negation. |
| **Duplicate Pairs** | 1 exact duplicate and 6 conflicting pairs removed for training view (`df_train_dedup`) | Eliminates contradictory gradient signals during training while preserving full raw data in memory. |
| **Validation Design** | 5-Fold `GroupKFold` grouped by `job_description_text_clean` | Strictly prevents job leakage across folds, perfectly simulating generalization to unseen jobs. |
| **Test Set Integrity** | `test.csv` held strictly read-only; no transformations fit on test set | Prevents data snooping and guarantees unbiased final evaluation. |

---

### 10.2 Viva & Project Defense Preparation (Q&A)

**Q1: Why did you deliberately avoid stop-word removal and Porter stemming?**  
*Answer:* In classical Information Retrieval, stemming and stop-word removal were used to reduce sparse vocabulary dimensionality. However, in modern NLP and Deep Learning (Sentence-BERT, Cross-Encoders), models rely on dense contextual representations. Removing stop words strips crucial context (e.g., distinguishing *"not qualified in Java"* from *"qualified in Java"*). Stemming collapses technical terms into invalid stems (e.g., *"React"* becomes *"react"*, losing framework identity). Our preprocessing cleans noise while preserving full linguistic and technical integrity.

**Q2: What is the risk of alphabetical label encoding, and how did you resolve it?**  
*Answer:* Alphabetical encoding maps `Good Fit` to 0, `No Fit` to 1, and `Potential Fit` to 2. This breaks ordinality because `No Fit` sits between the two positive/partial classes. We established an explicit, monotonic mapping: `No Fit` = 0, `Potential Fit` = 1, `Good Fit` = 2. This allows models to learn a natural candidate suitability progression and enables threshold-based screening in the web app.

**Q3: Why must cross-validation be grouped by job description instead of candidate resumes or random sampling?**  
*Answer:* The benchmark tests whether the model can score candidate resumes against **completely unseen job postings** (all 71 test jobs are disjoint from train). If random K-Fold were used, multiple candidate pairs for the same job description would appear in both training and validation folds. A neural model would memorize job-specific vocabulary and achieve falsely optimistic validation scores ("validation leakage"). Grouping by job description ensures every validation fold contains 100% unseen jobs, faithfully reflecting test set evaluation.

**Q4: How did you treat the 6 duplicate pairs with contradictory labels?**  
*Answer:* The training corpus contains 6 identical `(resume, job)` pairs assigned conflicting labels (e.g. `No Fit` in one row and `Good Fit` in another). Feeding identical inputs with contradictory supervisory targets creates conflicting gradient signals and introduces artificial noise. We flagged these rows transparently and filtered them out of the clean training set (`df_train_dedup`), reducing training size from 6,241 to 6,228 rows, while leaving the holdout test set completely untouched.

**Q5: Does text normalization automatically improve model performance?**  
*Answer:* No. Text normalization is an engineering safeguard to ensure consistency, eliminate corrupted font artifacts, and standardize structure. Whether normalization improves, maintains, or slightly affects downstream task accuracy must be validated experimentally during model evaluation in Stage 4C.

---

### 10.3 Next Steps: Stage 4C (Baseline Models)
In the next stage, we will proceed to baseline modeling:
1. **TF-IDF Vectorization:** Compute n-gram representations on `resume_text_clean` and `job_description_text_clean`.
2. **Heuristic Baseline:** Compute Cosine Similarity between resume and job TF-IDF vectors.
3. **Linear Baselines:** Train Logistic Regression and Linear Support Vector Classifiers (LinearSVC).
4. **Tree Baseline:** Train Gradient Boosted Trees (e.g., XGBoost / LightGBM) on concatenated TF-IDF or dense sentence embeddings.
5. **Evaluation Protocol:** Evaluate all models across the 5 disjoint folds established in this notebook using **Macro-averaged F1 Score** and per-class Precision/Recall.

---
*Notebook completed and verified for Verisift Stage 4B.*
